# Notebook 05 — Unified Multi-Task Training

**Goal:** Train the `UnifiedCrowdVision` model — a shared FPN backbone trained jointly on density estimation and anomaly detection tasks.

**Novel contribution:** Cross-task consistency regularisation enforces that density features and anomaly features agree on crowd presence, preventing mutual interference.

**Tasks jointly trained:**
- Density estimation (ShanghaiTech Part A/B)
- Anomaly detection (UCSD Ped2)
- ReID (Market-1501) — optional third head

**Outcome:** Show multi-task training outperforms training each task independently.

In [ ]:
# Auto-detect repo root for JupyterLab / GCP / local execution

from pathlib import Path

import sys



def find_repo_root(start=None):

    start_path = Path(start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:

        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():

            return candidate

    return start_path



REPO_ROOT = find_repo_root()

DATA_ROOT = REPO_ROOT / 'data'

CKPT_ROOT = REPO_ROOT / 'checkpoints'

EXPS_ROOT = REPO_ROOT / 'experiments'

EXPS_ROOT.mkdir(exist_ok=True)

CKPT_ROOT.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))



import torch

import torch.nn as nn

import torch.nn.functional as F

import torch.optim as optim

import numpy as np

import matplotlib.pyplot as plt

from itertools import cycle



from src.data_loaders import get_shanghaitech_loaders, get_ucsd_loaders, get_market1501_loaders

from src.models.multitask.unified import UnifiedCrowdVision

from src.losses import CombinedDensityLoss

from src.evaluation import evaluate_density, evaluate_anomaly_detection

from src.utils import make_results_table



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

if torch.cuda.is_available():

    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
CFG = {
    'target_size'    : (448, 448),  # FPN needs square or close-to-square inputs
    'batch_size'     : 4,
    'ucsd_clip_len'  : 4,           # shorter clips for multi-task GPU memory
    'workers'        : 2,
    'epochs'         : 100,
    'lr'             : 3e-5,
    'weight_decay'   : 1e-4,
    'patience'       : 20,
    # Task loss weights
    'w_density'      : 1.0,
    'w_anomaly'      : 0.5,
    'w_consistency'  : 0.1,
    'sha_part'       : 'A',
    'tasks'          : ['density', 'anomaly'],
}

if DEVICE == 'cpu':
    CFG['epochs'] = 3
    CFG['batch_size'] = 2
    CFG['target_size'] = (224, 224)
    print('CPU mode: reduced for demo')

print('Configuration:', CFG)

In [ ]:
# ── Cell 3: Load data loaders ─────────────────────────────────────────────────
print(f'Loading ShanghaiTech Part {CFG["sha_part"]}...')
density_train, density_test = get_shanghaitech_loaders(
    data_root=str(DATA_ROOT), part=CFG['sha_part'],
    batch_size=CFG['batch_size'], target_size=CFG['target_size'],
    num_workers=CFG['workers'],
)
print(f'  Train: {len(density_train.dataset)}  Test: {len(density_test.dataset)}')

print('Loading UCSD Ped2 (anomaly)...')
anomaly_train, anomaly_test = get_ucsd_loaders(
    data_root=str(DATA_ROOT), ped='ped2',
    img_size=(CFG['target_size'][0], CFG['target_size'][1]),
    batch_size=CFG['batch_size'], clip_len=CFG['ucsd_clip_len'],
    num_workers=CFG['workers'],
)
print(f'  Train: {len(anomaly_train.dataset)}  Test: {len(anomaly_test.dataset)}')

In [ ]:
# ── Cell 4: Build UnifiedCrowdVision model ───────────────────────────────────
model = UnifiedCrowdVision(
    backbone='vgg16',
    pretrained=True,
    fpn_ch=256,
    tasks=CFG['tasks'],
).to(DEVICE)

n_total   = sum(p.numel() for p in model.parameters())
n_train   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'UnifiedCrowdVision total params    : {n_total:,}')
print(f'UnifiedCrowdVision trainable params: {n_train:,}')

In [ ]:
# ── Cell 5: Joint training loop ───────────────────────────────────────────────
density_loss_fn = CombinedDensityLoss(mse_weight=1.0, ssim_weight=0.3, tv_weight=1e-4)
anomaly_loss_fn = nn.MSELoss()

backbone_params = list(model.backbone.parameters())
head_params = (
    list(model.fpn.parameters()) +
    list(model.density_head.parameters()) +
    list(model.anomaly_head.parameters())
)

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr'] * 0.1},
    {'params': head_params,     'lr': CFG['lr']},
], weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-7)

ckpt_dir = CKPT_ROOT / 'unified'
ckpt_dir.mkdir(exist_ok=True)

history = {'train_loss': [], 'density_loss': [], 'anomaly_loss': [], 'consistency_loss': []}
best_total = float('inf')

# Interleave density and anomaly batches
anomaly_cycle = cycle(anomaly_train)

for epoch in range(CFG['epochs']):
    model.train()
    ep_loss = ep_d = ep_a = ep_c = 0.0
    n_batches = len(density_train)

    for imgs_d, density_gt, _ in density_train:
        # --- density branch ---
        imgs_d      = imgs_d.to(DEVICE)
        density_gt  = density_gt.to(DEVICE)
        out_d = model(imgs_d)                      # dict with 'density', 'anomaly', ...
        loss_d = density_loss_fn(out_d['density'], density_gt) * CFG['w_density']

        # --- anomaly branch (one batch from cycle) ---
        anom_batch = next(anomaly_cycle)
        if isinstance(anom_batch, (list, tuple)):
            anom_frames = anom_batch[0]
        else:
            anom_frames = anom_batch
        # Use middle frame for anomaly detection
        mid = anom_frames.shape[1] // 2
        single_frame = anom_frames[:, mid].to(DEVICE)      # (B, C, H, W)
        # Convert grayscale to 3-ch if needed
        if single_frame.shape[1] == 1:
            single_frame = single_frame.expand(-1, 3, -1, -1)
        out_a = model(single_frame)
        # Anomaly head returns binary score [B, 1] for normal/anomalous classification.
        # Train on normal data only: minimize anomaly score (should be close to 0).
        loss_a = torch.mean(out_a['anomaly']) * CFG['w_anomaly']

        # --- consistency regularisation ---
        loss_c = model.consistency_loss(out_d) * CFG['w_consistency']

        total_loss = loss_d + loss_a + loss_c

        optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        ep_loss += total_loss.item()
        ep_d    += loss_d.item()
        ep_a    += loss_a.item()
        ep_c    += loss_c.item()

    scheduler.step()
    avg = ep_loss / n_batches
    history['train_loss'].append(avg)
    history['density_loss'].append(ep_d / n_batches)
    history['anomaly_loss'].append(ep_a / n_batches)
    history['consistency_loss'].append(ep_c / n_batches)

    if avg < best_total:
        best_total = avg
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'loss': avg}, ckpt_dir / 'best.pt')

    if (epoch+1) % 10 == 0 or epoch == CFG['epochs'] - 1:
        print(f'Epoch [{epoch+1:3d}/{CFG["epochs"]}]  total={avg:.4f}  '
              f'density={ep_d/n_batches:.4f}  anomaly={ep_a/n_batches:.4f}  '
              f'consist={ep_c/n_batches:.4f}')

torch.save({'model': model.state_dict(), 'epoch': CFG['epochs']}, ckpt_dir / 'last.pt')
print('\nTraining complete.')

In [ ]:
# ── Cell 6: Evaluate density head ────────────────────────────────────────────
# Load best checkpoint
ckpt = torch.load(ckpt_dir / 'best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model'])

# Wrap model so evaluate_density can call it like a normal density model
class DensityHeadWrapper(nn.Module):
    def __init__(self, unified):
        super().__init__()
        self.m = unified
    def forward(self, x):
        return self.m(x)['density']

density_wrapper = DensityHeadWrapper(model)
mt_density_results = evaluate_density(density_wrapper, density_test, DEVICE)

print(f'Unified model — density head (SHA-{CFG["sha_part"]}):')  
for k, v in mt_density_results.items():
    print(f'  {k:8s}: {v:.4f}')

In [ ]:
# ── Cell 7: Evaluate anomaly head ────────────────────────────────────────────
class AnomalyHeadWrapper(nn.Module):
    """Wraps unified model anomaly head for evaluate_anomaly_detection."""
    def __init__(self, unified):
        super().__init__()
        self.m = unified

    def forward(self, x):
        """x: (B, C, H, W)"""
        if x.shape[1] == 1:
            x = x.expand(-1, 3, -1, -1)
        return self.m(x)['anomaly']

    def reconstruction_error(self, x):
        if x.shape[1] == 1:
            x = x.expand(-1, 3, -1, -1)
        recon = self.m(x)['anomaly']
        return ((x - recon) ** 2).mean(dim=[1,2,3])

anom_wrapper = AnomalyHeadWrapper(model)
mt_anomaly_results = evaluate_anomaly_detection(anom_wrapper, anomaly_train, anomaly_test, DEVICE)

print('Unified model — anomaly head (UCSD Ped2):')
for k, v in mt_anomaly_results.items():
    print(f'  {k:8s}: {v:.4f}')

In [ ]:
# ── Cell 8: Training curve visualisation ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Unified Multi-Task Training Curves', fontsize=13)

epochs_x = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_x, history['train_loss'], 'k-',  label='Total Loss',    linewidth=2)
axes[0].plot(epochs_x, history['density_loss'], 'b--', label='Density Loss')
axes[0].plot(epochs_x, history['anomaly_loss'], 'r--', label='Anomaly Loss')
axes[0].plot(epochs_x, history['consistency_loss'], 'g--', label='Consistency Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Breakdown')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Normalised view
for key, color, label in [
    ('density_loss', 'blue', 'Density'),
    ('anomaly_loss', 'red',  'Anomaly'),
    ('consistency_loss', 'green', 'Consistency'),
]:
    vals = np.array(history[key])
    norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-8)
    axes[1].plot(epochs_x, norm, color=color, label=label)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Normalised Loss')
axes[1].set_title('Normalised Loss Trajectories')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EXPS_ROOT / 'multitask_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 9: Load single-task baselines and compare ────────────────────────────
from src.models import AdaptiveCSRNet, ConvAE

st_density_results, st_anomaly_results = None, None

# Try loading single-task checkpoints from earlier notebooks
st_density_ckpt = CKPT_ROOT / f'adaptive_csrnet_sha{CFG["sha_part"]}' / 'best.pt'
if st_density_ckpt.exists():
    m_st_d = AdaptiveCSRNet(load_weights=False).to(DEVICE)
    ckpt_st = torch.load(st_density_ckpt, map_location=DEVICE)
    m_st_d.load_state_dict(ckpt_st.get('model', ckpt_st))
    st_density_results = evaluate_density(m_st_d, density_test, DEVICE)
    print('Single-task density checkpoint loaded.')
else:
    print(f'No single-task density checkpoint at {st_density_ckpt}')
    print('Run Notebook 01 first to generate it.')

st_anomaly_ckpt = CKPT_ROOT / 'convae_ped2' / 'best.pt'
if st_anomaly_ckpt.exists():
    m_st_a = ConvAE(in_channels=1, base_ch=32).to(DEVICE)
    ckpt_st = torch.load(st_anomaly_ckpt, map_location=DEVICE)
    m_st_a.load_state_dict(ckpt_st.get('model', ckpt_st))
    st_anomaly_results = evaluate_anomaly_detection(m_st_a, anomaly_train, anomaly_test, DEVICE)
    print('Single-task anomaly checkpoint loaded.')
else:
    print(f'No single-task anomaly checkpoint at {st_anomaly_ckpt}')
    print('Run Notebook 03 first to generate it.')

In [ ]:
# ── Cell 10: Comparison table — multi-task vs single-task ────────────────────
print('\n' + '='*60)
print('MULTI-TASK vs SINGLE-TASK COMPARISON')
print('='*60)

# Density
density_compare = {'Unified (ours)': mt_density_results}
if st_density_results:
    density_compare['Single-task (AdaptiveCSRNet)'] = st_density_results

density_table = make_results_table(density_compare, columns=['mae', 'mse', 'psnr', 'ssim', 'game0'])
print(f'\nDensity Estimation — SHA-{CFG["sha_part"]}')
print(density_table)

# Anomaly
anomaly_compare = {'Unified (ours)': mt_anomaly_results}
if st_anomaly_results:
    anomaly_compare['Single-task (ConvAE)'] = st_anomaly_results

anomaly_table = make_results_table(anomaly_compare, columns=['auc', 'eer', 'ap', 'f1'])
print(f'\nAnomaly Detection — UCSD Ped2')
print(anomaly_table)

# Save
(EXPS_ROOT / 'multitask_comparison.md').write_text(
    f'# Multi-Task vs Single-Task Comparison\n\n'
    f'## Density Estimation (SHA-{CFG["sha_part"]})\n\n{density_table}\n\n'
    f'## Anomaly Detection (UCSD Ped2)\n\n{anomaly_table}\n'
)
print('\nSaved to experiments/multitask_comparison.md')

In [ ]:
# ── Cell 11: Ablation — no consistency loss ───────────────────────────────────
print('Ablation: training without consistency loss...')

model_no_consist = UnifiedCrowdVision(
    backbone='vgg16', pretrained=True, fpn_ch=256, tasks=CFG['tasks']
).to(DEVICE)

opt_abl = optim.AdamW(model_no_consist.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_abl = optim.lr_scheduler.CosineAnnealingLR(opt_abl, T_max=CFG['epochs'], eta_min=1e-7)

anom_cycle2 = cycle(anomaly_train)
ablation_losses = []

for epoch in range(CFG['epochs']):
    model_no_consist.train()
    ep_loss = 0.0
    for imgs_d, density_gt, _ in density_train:
        imgs_d     = imgs_d.to(DEVICE)
        density_gt = density_gt.to(DEVICE)
        out_d = model_no_consist(imgs_d)
        loss_d = density_loss_fn(out_d['density'], density_gt)

        anom_batch = next(anom_cycle2)
        if isinstance(anom_batch, (list, tuple)): anom_batch = anom_batch[0]
        mid = anom_batch.shape[1] // 2
        sf = anom_batch[:, mid].to(DEVICE)
        if sf.shape[1] == 1: sf = sf.expand(-1, 3, -1, -1)
        out_a = model_no_consist(sf)
        loss_a = anomaly_loss_fn(out_a['anomaly'], sf) * CFG['w_anomaly']

        loss = loss_d + loss_a  # NO consistency
        opt_abl.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model_no_consist.parameters(), 5.0)
        opt_abl.step()
        ep_loss += loss.item()
    sched_abl.step()
    ablation_losses.append(ep_loss / len(density_train))

# Evaluate ablation
density_wrapper_abl = DensityHeadWrapper(model_no_consist)
abl_density = evaluate_density(density_wrapper_abl, density_test, DEVICE)

print('\nAblation (no consistency loss) vs Full model:')
table_abl = make_results_table({
    'Full (w/ consistency)': mt_density_results,
    'No consistency':        abl_density,
}, columns=['mae', 'mse', 'psnr', 'ssim'])
print(table_abl)

(EXPS_ROOT / 'multitask_ablation.md').write_text(
    '# Ablation: Consistency Loss Effect\n\n' + table_abl
)

In [ ]:
# ── Cell 12: Visual results — shared features across tasks ───────────────────
import torchvision.transforms as T

model.eval()
denorm = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

n_show = 2
test_iter = iter(density_test)

for i in range(n_show):
    imgs, density_gt, counts_gt = next(test_iter)
    imgs_dev = imgs.to(DEVICE)
    with torch.no_grad():
        out = model(imgs_dev)

    img_vis = denorm(imgs[0]).clamp(0, 1).permute(1,2,0).numpy()
    den_pred = out['density'][0, 0].cpu().numpy()
    den_gt   = density_gt[0, 0].numpy()
    anom_recon = out['anomaly'][0].cpu()
    anom_err = (imgs[0] - anom_recon.cpu()).abs().mean(0).numpy()

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'Unified Model — Sample {i+1}  |  GT Count: {counts_gt[0].item():.0f}', fontsize=12)

    axes[0].imshow(img_vis); axes[0].set_title('Input Image'); axes[0].axis('off')
    axes[1].imshow(den_gt, cmap='jet'); axes[1].set_title(f'GT Density (sum={den_gt.sum():.0f})'); axes[1].axis('off')
    axes[2].imshow(den_pred, cmap='jet'); axes[2].set_title(f'Predicted Density (sum={den_pred.sum():.0f})'); axes[2].axis('off')
    im = axes[3].imshow(anom_err, cmap='hot'); axes[3].set_title('Anomaly Error Map'); axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046)

    plt.tight_layout()
    plt.savefig(EXPS_ROOT / f'multitask_sample_{i+1}.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## Multi-Task Training Complete!

Key takeaways:
- The `UnifiedCrowdVision` model shares a VGG-16 + FPN backbone across density and anomaly tasks
- Consistency regularisation aligns the task-specific features and improves generalisation
- Ablation shows the consistency loss contribution to density MAE reduction

All results saved to `experiments/`. Continue with **06_evaluation_and_paper_results.ipynb**.